# KÜS source adapter

Control panel for the KÜS adapter (`kues` package: `1_scrapers/kues/src/`). All logic lives in the modules (`fetch`, `parse`, `classify`, `run`, `report`, shared `core.radius_cover`); this notebook only calls them, and every path comes from `core.paths`. Recon notes, field table and the coverage design with numbers: `0_instructions/sources/kues.md`. KÜS needs **no key and no cookies**.

**How the finder works.** `GET webservice.kues.de/api/kues/partnersuche?plz=<PLZ>` returns every Partner within **50 km** of the PLZ (fixed radius, **no count field**, no paging). So completeness is *proved* instead of read from a total, with four rules on the **actual server centres**:

| Rule | Check |
|---|---|
| **A** coverage | every PLZ centroid (8,309) lies within `COVER_KM` of a server centre |
| **B** fringe | every collected site lies within 45 km of a centre (else its own PLZ is queried) |
| **C** consistency | every centre within 48 km of a collected site returned that site (a miss = cap, flaky server or changed radius) |
| **D** cap detector | no response at 500 records, no repeated maximum count |

**Scope decision (2026-10-05, extended 2026-10-07).** `pruefungvorortmoeglich = 1` → `pruefstelle`, `in_scope = true`; `0` → `partner_office` (inspects at PSPs, out of scope). Everything is collected; the export `kues_pruefstellen_<date>.csv` is the `in_scope` filter of `kues_all_<date>.csv`. **Exception (Nikos, 2026-10-07: no certain cause to exclude → include):** a `0` record whose own name says Prüfstelle / Prüfzentrum / Prüfingenieur is kept (`site_type = unknown`, `in_scope = true`, `needs_review`), except the three that were checked by hand and are not Prüfstellen (`classify.VERIFIED_NOT_PRUEFSTELLE`, shown with that reason in `scope_reason`). Other conflicts (flag 1 without address) are **flagged for review**, never overruled.

## 1. Setup and parameters

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from core import paths, radius_cover as rc
from core.geo import ensure_geonames, load_plz_centroids
from kues import fetch, report
from kues import run as kues_run

ROOT = paths.project_root()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# ---- parameters (plain language) -------------------------------------------------------------------------------
RADIUS_KM = 50          # the finder's search radius. 50 is what the web page uses. The server also accepts up to 100 through an
                        # undocumented `km` parameter (about 4x fewer requests); untested at scale, leave at 50.
COVER_KM = 25           # rule A: every PLZ centre must lie within this many km of a real server centre. RADIUS_KM minus COVER_KM is the
                        # room for the distance between a site and the centre of its own PLZ (measured maximum: 12.8 km).
FETCH_TECHNIK = True    # also ask the "KÜS Technik" list (a subset of the main list; gives kues_list = both). False halves the requests.
REQUEST_LIMIT = fetch.FULL_MAX_REQUESTS   # hard ceiling of live requests in a full run; the run stops at it
EXPORTS = ("all_csv", "all_xlsx", "pruefstellen_csv", "pruefstellen_xlsx")   # which export files a run writes
SAVE_RAW = True         # raw.jsonl keeps every response (~25 MB for a full run); without it a run cannot be rebuilt offline
SAMPLE_PLZ = "80636"    # the PLZ of the sample run (München)

cfg = rc.CoverConfig.for_radius(RADIUS_KM, cover_km=COVER_KM)
centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
print(f"{len(centroids)} regular PLZ centroids (GeoNames, CC BY 4.0)")
cfg

## 2. Before the full run: plan, request count, runtime

The planned queries are a hexagonal lattice over Germany, each point snapped to a real PLZ. The **expected total** adds the fill-in queries that rules A and B will need; it comes from a simulated crawl with server-centre offsets like the real ones (0–6 km), so the real number can differ. The runtime assumes 3 s pause + up to 1 s jitter + ~0.8 s per answer.

In [ ]:
est = kues_run.estimate(centroids, cfg, fetch_technik=FETCH_TECHNIK)
print(f"planned queries (PLZ):     {est['planned_queries']}")
print(f"expected incl. fill-in:    {est['expected_queries']} (fill-in ~{est['expected_fill_in']})")
print(f"lists per query:           {est['lists']}")
print(f"expected requests:         {est['expected_requests']}  (hard limit {REQUEST_LIMIT})")
print(f"expected runtime:          ~{est['expected_runtime_min']} min")
report.planned_map(centroids, est["planned_points"], cfg);

## 3. Runs

Every run writes one folder `YYYY-MM-DD_HHMM_<mode>` (`full` and `rebuild` in `1_scrapers/kues/output/`, the rest in `test_output/`) with `query_log.csv`, `run_summary.json`/`.md`, the exports and, if `SAVE_RAW`, `raw.jsonl`. A run never writes into an existing folder.

**Run status.** PASS (full) = no error or abort, every planned query made, rules A–D hold (technik list: C and D), drop check ok. A run that is not a PASS labels its exports `grid-incomplete` (aborted, planned queries missing, rule A/B) or `unverified` (rule C/D, drop alert). Simulated, sample and recon runs never count as verified crawls. The drop check compares the in-scope count with the newest earlier **passing** full run (alert below −10 %).

**Politeness.** Sequential, 3 s + jitter between requests, 30 s timeout, 3 attempts with backoff on timeouts/5xx; the run stops on 401/403, a second 429 in a row, a non-JSON answer, an unexpected response shape and at the request limit.

### 3a. Simulated run (offline, default)

~1,500 fake sites over Germany behind a fake API with server centres offset like the real ones. Must **PASS** and find 100 % of the sites.

In [ ]:
sim = kues_run.run(mode="simulated", centroids=centroids, cfg=cfg, fetch_technik=FETCH_TECHNIK, exports=EXPORTS, save_raw=False)
RESULT = sim   # the run the sections below show; set RESULT = full after a full run, or RESULT = report.load_run("<run folder>")

### 3b. Recon (offline)

Parses a saved response of one PLZ (here the München response saved on 2026-10-05) into a `recon` run folder. No requests.

In [ ]:
SAVED = paths.reference_dir() / "api_samples" / "kues" / "partnersuche_80636_2026-10-05.json"
recon = kues_run.run(mode="recon", centroids=centroids, cfg=cfg, source_file=SAVED, exports=("pruefstellen_csv",))

### 3c. Rebuild (offline)

Rebuilds the exports and re-checks rules A–D from the `raw.jsonl` of an earlier run (needs `SAVE_RAW`). The new folder goes to `output/`, no copy of the raw file. Set `REBUILD_FROM` to a run folder name or path; the default is skipped.

In [ ]:
REBUILD_FROM = None   # e.g. "2026-10-06_0900_full"
if REBUILD_FROM:
    rebuilt = kues_run.run(mode="rebuild", centroids=centroids, cfg=cfg, source_run=REBUILD_FROM, exports=EXPORTS)
else:
    print("Rebuild skipped: set REBUILD_FROM to a run folder.")

### 3d. Sample run (LIVE, 2 requests)

**Live requests.** Queries only `SAMPLE_PLZ` (München 80636): the main list and, with `FETCH_TECHNIK`, the technik list. It compares the answer with the saved response of the same PLZ and never reports PASS. Skipped unless `RUN_LIVE_SAMPLE = True`.

In [ ]:
RUN_LIVE_SAMPLE = False   # True = make 2 live requests
if RUN_LIVE_SAMPLE:
    sample = kues_run.run(mode="sample", centroids=centroids, cfg=cfg, sample_plz=SAMPLE_PLZ, fetch_technik=FETCH_TECHNIK, exports=EXPORTS, save_raw=SAVE_RAW)
else:
    print("Sample run skipped: set RUN_LIVE_SAMPLE = True to make 2 live requests.")

### 3e. Full run (LIVE)

**Live requests: the expected number is in section 2** (about 700 with both lists, ~50 minutes). KÜS needs no key and automated collection is covered by the management approval of 2026-09-28. The cell does nothing unless `confirm_full_run = True`; keep it `False` in the committed notebook. When it finishes the cell sets `RESULT = full`, so section 4 and the sections after it show the full run.

In [ ]:
confirm_full_run = False   # True = start the full run (live requests)
if confirm_full_run:
    full = kues_run.run(
        mode="full", confirm_full_run=True, centroids=centroids, cfg=cfg, fetch_technik=FETCH_TECHNIK,
        max_requests=REQUEST_LIMIT, exports=EXPORTS, save_raw=SAVE_RAW,
    )
    RESULT = full
else:
    print("Full run NOT started: set confirm_full_run = True to make about", est["expected_requests"], "live requests.")

## 4. Results

Shows `RESULT` (the simulated run by default; after a full run set `RESULT = full`). Counts and maps contain no personal data for simulated runs; for real runs the review tables (section 6) show names: **do not save the notebook with those outputs**.

### 4a. Outcome and coverage rules

In [ ]:
s = RESULT.summary
print(s["outcome"], "|", RESULT.run_folder.name)
for reason in s["status_reasons"]:
    print("  !", reason)
print("requests:", s["requests_made"], "| plan:", s["plan"])
report.rule_table(s)

### 4b. Counts, per Bundesland, drop check

In [ ]:
display(report.counts_table(RESULT).to_frame("count"))
print("Drop check:", s["drop_check"])
if s.get("baseline"):
    print("Baseline:  ", s["baseline"])
print("Flags:", s["flags"])
report.per_state(RESULT.frame)

### 4c. Published KÜS figures

What KÜS says about its network, and what we found. Only the first claim was found on a page (people, not sites); the second is unverified.

In [ ]:
report.published_comparison(s)

### 4d. Warnings (not failures)

Empty responses, server centres far from the GeoNames centroid, a radius that looks different from 50 km, content conflicts for the same `adressid`.

In [ ]:
for name, items in (s.get("warnings") or {}).items():
    print(f"{name}: {len(items)}")
    for item in items[:5]:
        print("   ", item)
if not s.get("warnings"):
    print("no warnings")

## 5. Maps

In [ ]:
report.coverage_map(RESULT, centroids, cfg);   # server centres with their search circles; black x = PLZ not covered / answered empty

In [ ]:
report.sites_map(RESULT.frame);   # red = in scope (flag 1, plus flag-0 records whose name says Prüfstelle), grey = partner_office

## 6. Review tables (contain names for real runs)

Everything stays in `kues_all`; these are the rows a human should look at.

In [ ]:
tables = report.review_tables(RESULT.frame)
tables["needs_review_by_reason"]

In [ ]:
tables["flag_conflicts"].head(20)   # name vs flag conflicts and flag-1-without-address rows

In [ ]:
tables["licensees_with_several_sites"].head(20)   # one licence code, several sites: separate rows, never merged

In [ ]:
tables["shared_addresses"].head(20)   # same street + PLZ, several records: never deduped by address

## 7. Stage 2: pattern analysis (2026-10-05)

An independent subagent analysed the München sample (83 records) and the recon responses (several hundred distinct sites) against the decision rule. Full report: `1_scrapers/kues/test_output/2026-10-05_1635_sample/pattern_analysis.md` (contains personal data, git-ignored). Spot-check list for Nikos: `4_reference/ground_truth/kues_spotcheck_2026-10-05.csv`. Conclusions from one region need the full run before they can be trusted.

**Findings (449 distinct sites: München sample plus the recon responses; not a random sample, not ground truth):**

- The flag looks plausible but is unproven: 52 of 55 records with *Prüfstelle/Prüfzentrum* in the name have flag 1; private email addresses are 52 % of flag 0 vs 19 % of flag 1.
- All 22 `keinestrasse = 1` records have flag 0 (cut, no review). Flag 1 always has street and coordinates.
- The weak side is flag 1: 158 of 222 have no station word in the name (82 are "Ingenieurbüro"); 59 have no station word, no website and no business email.
- Name conflicts on the flag-0 side are rare (4 of 227); 42 flag-0 records share a licence code with a flag-1 site; 6 are near-identical twins of a flag-1 record.
- 11 shared-address groups, multi-site licensees (SV Team AG, Has4, SVZ, PWE, KPN) are separate addresses: never merge by licence code or address.
- No PSP-like names anywhere: the finder lists licensees, not the 8,000 PSPs.
- The technik list adds no scope information (subset of the main list).
- 7 records (licence I1122, Has4 Technologies, NRW) use the HolDeinePlakette email domain: probably HDP's own sites.

**Implemented as flags and columns (nothing dropped, no scope effect):** `vorort1_weak_evidence`, `vorort0_strong_evidence`, `contact_missing`, `same_address_as_inscope_site`, `shared_address_in_scope`, `name_psp_like` (needs_review), columns `name_class` and `licensee_has_inscope_site`.

**Needs a decision (not implemented):** is a flag-1 office without station evidence a Prüfstelle; what to do with flag-0 twins and siblings; HubSpot company granularity; KÜS-owned units; flag-1 rows without contact data; keep the technik query; chains. The full list of nine questions is in `0_instructions/sources/kues.md` §5b.